In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nest
from scipy.stats import truncnorm

nest.ResetKernel()


             -- N E S T --

 Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0-post0.dev14
 Built  : Mar  5 2026 15:42:06

 This program is provided AS IS and comes with NO WARRANTY.
 See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [3]:
truncnorm.rvs(a=1, b=2, size = 10)

array([1.47237836, 1.72058411, 1.2932912 , 1.05252801, 1.21659249,
       1.19294496, 1.2323617 , 1.22106342, 1.10010652, 1.23781772])

In [11]:
data_path = "data/"

data = np.loadtxt(data_path + "network_structure.dat")

# the id of the neurons are from 2 to 1001
data_filtered = data[(data[:, 1] <= 1001)]

ordered_ids = np.argsort(data_filtered[:,0])
data_ordered = data_filtered[ordered_ids]

# separate the exc to exc connection from the others
# the first one are connected using stp, the second one are connected using static synapses
mask = (data_ordered[:,0] <= 801) & (data_ordered[:,1] <= 801)
data_excTOexc = data_ordered[mask]
data_other = data_ordered[~mask]

sources_excTOexc = data_excTOexc[:, 0].astype(int) - 1
targets_excTOexc = data_excTOexc[:, 1].astype(int) - 1
weights_excTOexc = data_excTOexc[:, 2]
delays_excTOexc = data_excTOexc[:, 3]

sources_other = data_other[:, 0].astype(int) - 1
targets_other = data_other[:, 1].astype(int) - 1
weights_other = data_other[:, 2]
delays_other = data_other[:, 3]

In [98]:
num_neurons = 1000
exc_neurons = nest.Create("iaf_psc_exp", 800)
inh_neurons = nest.Create("iaf_psc_exp", 200)

all_neurons = exc_neurons + inh_neurons

In [99]:
N_sel = 80        # Dimensione di ciascuna delle 5 popolazioni selettive
N_nonsel = 400    # Dimensione della popolazione eccitatoria non selettiva
N_inh = 200       # Dimensione della popolazione inibitoria

In [100]:
pop_ecc_selettive = [all_neurons[i * N_sel : (i + 1) * N_sel] for i in range(5)]
pop_ecc_non_selettiva = all_neurons[5 * N_sel : 5 * N_sel + N_nonsel]
pop_inibitoria = all_neurons[5 * N_sel + N_nonsel : num_neurons]

In [101]:
nest.Connect(
    file_sources, 
    file_targets, 
    conn_spec={'rule': 'one_to_one'},
    syn_spec={
        'synapse_model': 'static_synapse',
        'weight': weights,
        'delay': delays
    }
)